# Análisis Integral de Ventas - Maven Roasters

**Periodo de análisis:** enero-junio de 2023  
**Objetivo:** profundizar el análisis exploratorio del desempeño transaccional de Maven Roasters, ampliando el cuaderno existente con una estructura más rigurosa, interpretaciones ejecutivas y estandarización del código.

---


### 1. Introducción

Este cuaderno retoma el trabajo preliminar ya desarrollado para Maven Roasters y lo extiende con un análisis exploratorio mucho más profundo. El propósito es convertir el notebook en un documento analítico de lectura ejecutiva: cada bloque combina código, tablas, visualizaciones e interpretación en español formal.

El análisis se orienta a responder tres preguntas de negocio: cómo se distribuyen las ventas, qué variables describen mejor el comportamiento del portafolio y qué relaciones entre atributos de producto, tiempo y tienda generan diferencias observables en cantidad, precio e ingreso por transacción.


### 2. Dataset overview

Primero se inspecciona la estructura del archivo fuente, se valida la hoja utilizada y se generan variables analíticas derivadas que facilitan una exploración más rica sin alterar la integridad de los datos originales.


In [ ]:
import warnings
import json
import nbformat
import numpy as np
import pandas as pd
from textwrap import dedent
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display
from plotly.subplots import make_subplots
from wordcloud import WordCloud

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

PALETA = ['#0466c8', '#0353a4', '#023e7d', '#002855', '#001845', '#001233', '#33415c', '#5c677d', '#7d8597', '#979dac']
RUTA_ARCHIVO = '/home/ubuntu/Uploads/coffee_shop_sales.xlsx'
RUTA_NOTEBOOK = '/home/ubuntu/maven_roasters_analisis_completo.ipynb'

MESES = {1:'Enero',2:'Febrero',3:'Marzo',4:'Abril',5:'Mayo',6:'Junio',7:'Julio',8:'Agosto',9:'Septiembre',10:'Octubre',11:'Noviembre',12:'Diciembre'}
DIAS = {0:'Lunes',1:'Martes',2:'Miércoles',3:'Jueves',4:'Viernes',5:'Sábado',6:'Domingo'}


def estilizar_figura(fig, titulo, x=None, y=None, leyenda=None, alto=520):
    fig.update_layout(
        title=dict(text=titulo, x=0.5, xanchor='center', font=dict(family='Arial Black', size=20)),
        font=dict(family='Arial', size=12),
        colorway=PALETA,
        template='plotly_white',
        height=alto,
        legend_title_text=leyenda,
        margin=dict(l=40, r=40, t=90, b=40)
    )
    if x:
        fig.update_xaxes(title_text=x)
    if y:
        fig.update_yaxes(title_text=y)
    return fig


def resumen_textual_serie(serie):
    return {
        'n': int(serie.shape[0]),
        'media': float(serie.mean()),
        'mediana': float(serie.median()),
        'q1': float(serie.quantile(0.25)),
        'q3': float(serie.quantile(0.75)),
        'min': float(serie.min()),
        'max': float(serie.max()),
        'sesgo': float(serie.skew()) if serie.nunique() > 2 else np.nan
    }


def tabla_iqr(serie):
    q1 = serie.quantile(0.25)
    mediana = serie.quantile(0.50)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    regiones = pd.Series(index=serie.index, dtype='object')
    regiones[serie < lim_inf] = 'Por debajo del bigote inferior'
    regiones[(serie >= lim_inf) & (serie < q1)] = 'Entre bigote inferior y Q1'
    regiones[(serie >= q1) & (serie < mediana)] = 'Entre Q1 y mediana'
    regiones[(serie >= mediana) & (serie < q3)] = 'Entre mediana y Q3'
    regiones[(serie >= q3) & (serie <= lim_sup)] = 'Entre Q3 y bigote superior'
    regiones[serie > lim_sup] = 'Por encima del bigote superior'
    tabla = regiones.value_counts(dropna=False).rename_axis('Rango IQR').reset_index(name='Conteo')
    orden = ['Por debajo del bigote inferior','Entre bigote inferior y Q1','Entre Q1 y mediana','Entre mediana y Q3','Entre Q3 y bigote superior','Por encima del bigote superior']
    tabla['Rango IQR'] = pd.Categorical(tabla['Rango IQR'], categories=orden, ordered=True)
    tabla = tabla.sort_values('Rango IQR').reset_index(drop=True)
    tabla['Porcentaje'] = (tabla['Conteo'] / len(serie) * 100).round(2)
    return tabla, regiones, {'q1': q1, 'mediana': mediana, 'q3': q3, 'iqr': iqr, 'lim_inf': lim_inf, 'lim_sup': lim_sup}


def insight_numerico(nombre, serie, tabla_regiones):
    stats = resumen_textual_serie(serie)
    mayor_bloque = tabla_regiones.sort_values('Conteo', ascending=False).iloc[0]
    outliers_altos = float(tabla_regiones.loc[tabla_regiones['Rango IQR'] == 'Por encima del bigote superior', 'Porcentaje'].fillna(0).sum())
    outliers_bajos = float(tabla_regiones.loc[tabla_regiones['Rango IQR'] == 'Por debajo del bigote inferior', 'Porcentaje'].fillna(0).sum())
    sesgo = 'asimetría positiva' if stats['media'] > stats['mediana'] else 'asimetría negativa' if stats['media'] < stats['mediana'] else 'distribución aproximadamente simétrica'
    return dedent(f"""
    **Lectura ejecutiva.** La variable **{nombre}** presenta una media de **{stats['media']:.2f}** y una mediana de **{stats['mediana']:.2f}**, lo que sugiere **{sesgo}**. El tramo con mayor concentración es **{mayor_bloque['Rango IQR']}** con **{mayor_bloque['Porcentaje']:.2f}%** de las observaciones. Los valores extremos representan **{outliers_bajos:.2f}%** por debajo y **{outliers_altos:.2f}%** por encima de los bigotes, por lo que la variable puede contener eventos poco frecuentes pero potencialmente relevantes para la operación.
    """).strip()


def frecuencia_categorica(df, columna):
    tabla = df[columna].astype(str).value_counts(dropna=False).rename_axis(columna).reset_index(name='Conteo')
    tabla['Porcentaje'] = (tabla['Conteo'] / len(df) * 100).round(2)
    return tabla


def tabla_para_grafico_categorico(tabla, columna, max_categorias=12):
    if len(tabla) <= max_categorias:
        return tabla.copy()
    top = tabla.head(max_categorias).copy()
    otros = pd.DataFrame({
        columna: ['Otros'],
        'Conteo': [tabla.iloc[max_categorias:]['Conteo'].sum()],
        'Porcentaje': [tabla.iloc[max_categorias:]['Porcentaje'].sum()]
    })
    return pd.concat([top, otros], ignore_index=True)


def columna_apta_wordcloud(serie):
    n_unicos = serie.nunique(dropna=False)
    promedio_largo = serie.astype(str).str.len().mean()
    return n_unicos <= 120 and promedio_largo >= 3


def insight_categorico(columna, tabla):
    top = tabla.iloc[0]
    diversidad = tabla[columna].nunique()
    return dedent(f"""
    **Lectura ejecutiva.** La variable **{columna}** contiene **{diversidad}** categorías observadas. La categoría más frecuente es **{top[columna]}**, con **{top['Conteo']:,} registros** (**{top['Porcentaje']:.2f}%** del total). La distribución permite evaluar concentración operativa, amplitud del portafolio y posibles dependencias del negocio respecto de unas pocas categorías dominantes.
    """).strip()


def resumen_relacion(df, cat_col, num_col):
    agrupado = df.groupby(cat_col, dropna=False)[num_col].agg(['count','mean','median','min','max']).reset_index()
    agrupado = agrupado.sort_values('mean', ascending=False)
    agrupado.columns = [cat_col, 'conteo', 'media', 'mediana', 'mínimo', 'máximo']
    return agrupado


def insight_relacion(cat_col, num_col, tabla):
    alto = tabla.iloc[0]
    bajo = tabla.iloc[-1]
    amplitud = alto['media'] - bajo['media']
    return dedent(f"""
    **Lectura ejecutiva.** La relación entre **{cat_col}** y **{num_col}** muestra diferencias visibles entre grupos. La categoría con mayor promedio es **{alto[cat_col]}** (**{alto['media']:.2f}**), mientras que la menor corresponde a **{bajo[cat_col]}** (**{bajo['media']:.2f}**). La brecha promedio de **{amplitud:.2f}** sugiere una asociación útil para segmentación comercial, priorización de portafolio o ajuste operativo.
    """).strip()


xls = pd.ExcelFile(RUTA_ARCHIVO)
hojas = xls.sheet_names
raw_df = pd.read_excel(RUTA_ARCHIVO, sheet_name=hojas[0])
df = raw_df.copy()
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df['transaction_time'] = pd.to_datetime(df['transaction_time'].astype(str), format='%H:%M:%S')
df['revenue'] = df['transaction_qty'] * df['unit_price']
df['year'] = df['transaction_date'].dt.year
df['month'] = df['transaction_date'].dt.month
df['day'] = df['transaction_date'].dt.day
df['day_of_week'] = df['transaction_date'].dt.dayofweek
df['hour'] = df['transaction_time'].dt.hour
df['month_name'] = pd.Categorical(df['month'].map(MESES), categories=['Enero','Febrero','Marzo','Abril','Mayo','Junio'], ordered=True)
df['day_name'] = pd.Categorical(df['day_of_week'].map(DIAS), categories=['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo'], ordered=True)
df['weekend_flag'] = np.where(df['day_of_week'] >= 5, 'Fin de semana', 'Día laboral')
df['time_of_day'] = pd.Categorical(pd.cut(df['hour'], bins=[5,10,13,16,20], labels=['Mañana','Mediodía','Tarde','Noche']), categories=['Mañana','Mediodía','Tarde','Noche'], ordered=True)

display(Markdown('#### Confirmación de carga y preparación analítica'))
resumen_archivo = pd.DataFrame({
    'Elemento': ['Archivo fuente', 'Hojas detectadas', 'Hoja utilizada', 'Filas', 'Columnas originales', 'Columnas analíticas finales', 'Fecha mínima', 'Fecha máxima', 'Ingresos totales'],
    'Valor': [RUTA_ARCHIVO, len(hojas), hojas[0], f"{raw_df.shape[0]:,}", raw_df.shape[1], df.shape[1], str(df['transaction_date'].min().date()), str(df['transaction_date'].max().date()), f"USD {df['revenue'].sum():,.2f}"]
})
display(resumen_archivo)
display(Markdown('**Insight.** El archivo contiene una sola hoja transaccional y, tras la ingeniería mínima de variables, el conjunto queda listo para un EDA más explicativo sin perder trazabilidad respecto de las columnas originales.'))

display(Markdown('#### Vista preliminar de registros'))
display(df.head(10))


### 3. Variable typing and classification

En esta sección se clasifican las variables originales y las derivadas en roles analíticos. Se separan explícitamente los identificadores —incluidos los que son numéricos— para evitar interpretaciones engañosas en correlaciones o relaciones estadísticas.


In [ ]:
registros = []
columnas_originales = list(raw_df.columns)
for col in columnas_originales:
    dtype = str(raw_df[col].dtype)
    if col in ['transaction_id', 'store_id', 'product_id']:
        clasificacion = 'Identificador'
        rol = 'Llave operativa'
        incluir = 'Excluir de correlación y relaciones estadísticas'
        nota = 'Es numérica por codificación, no por significado analítico.'
    elif 'date' in col:
        clasificacion = 'Fecha'
        rol = 'Temporal'
        incluir = 'Usar para derivar mes, día y estacionalidad'
        nota = 'Se analiza mejor mediante variables temporales derivadas.'
    elif 'time' in col:
        clasificacion = 'Hora'
        rol = 'Temporal'
        incluir = 'Usar para derivar hora y franja horaria'
        nota = 'Se excluye de correlación directa por formato temporal.'
    elif pd.api.types.is_numeric_dtype(raw_df[col]):
        clasificacion = 'Numérica'
        rol = 'Métrica'
        incluir = 'Incluir'
        nota = 'Variable cuantitativa interpretable para distribución y relaciones.'
    else:
        clasificacion = 'No numérica'
        rol = 'Categórica'
        incluir = 'Incluir'
        nota = 'Apta para análisis de frecuencias y segmentación.'
    registros.append({'Variable': col, 'Origen': 'Original', 'dtype': dtype, 'Clasificación': clasificacion, 'Rol analítico': rol, 'Regla de uso': incluir, 'Nota': nota})

derivadas = {
    'revenue': ('Numérica', 'Métrica de negocio', 'Incluir', 'Ingreso por línea transaccional; variable clave para interpretación comercial.'),
    'year': ('Numérica', 'Temporal derivada', 'Excluir', 'Constante en este dataset; no aporta variabilidad analítica.'),
    'month': ('Numérica', 'Temporal derivada', 'Excluir', 'Representa orden temporal, pero se interpreta mejor con month_name.'),
    'day': ('Numérica', 'Temporal derivada', 'Excluir', 'Ordinal de calendario; útil descriptivamente, no para correlación principal.'),
    'day_of_week': ('Numérica', 'Temporal derivada', 'Excluir', 'Codificación ordinal del día; se utiliza day_name para relaciones.'),
    'hour': ('Numérica', 'Temporal derivada', 'Incluir', 'Aproxima el momento del día y sí tiene lectura operativa.'),
    'month_name': ('No numérica', 'Temporal categórica', 'Incluir', 'Facilita lectura ejecutiva del patrón mensual.'),
    'day_name': ('No numérica', 'Temporal categórica', 'Incluir', 'Facilita comparación semanal.'),
    'weekend_flag': ('No numérica', 'Temporal categórica', 'Incluir', 'Resume comportamiento entre días laborales y fin de semana.'),
    'time_of_day': ('No numérica', 'Temporal categórica', 'Incluir', 'Resume el patrón intradía en franjas operativas.')
}
for col, valores in derivadas.items():
    registros.append({'Variable': col, 'Origen': 'Derivada', 'dtype': str(df[col].dtype), 'Clasificación': valores[0], 'Rol analítico': valores[1], 'Regla de uso': valores[2], 'Nota': valores[3]})

clasificacion_df = pd.DataFrame(registros)
display(clasificacion_df)

elegibles_numericas = ['transaction_qty', 'unit_price', 'revenue', 'hour']
elegibles_categoricas = ['store_location', 'product_category', 'product_type', 'product_detail', 'month_name', 'day_name', 'weekend_flag', 'time_of_day']
excluidas_analisis = ['transaction_id', 'store_id', 'product_id', 'transaction_date', 'transaction_time', 'year', 'month', 'day', 'day_of_week']

display(Markdown(f"**Insight.** Se identifican **{len(elegibles_numericas)} variables numéricas elegibles** y **{len(elegibles_categoricas)} variables no numéricas elegibles** para el EDA profundo. Las variables excluidas del análisis relacional principal son: **{', '.join(excluidas_analisis)}**."))


### 4. Global numeric analysis (correlation matrix, pairplot for relevant numeric variables only)

El análisis global se limita a variables numéricas con significado analítico directo. Se excluyen identificadores y códigos temporales para evitar correlaciones espurias.


In [ ]:
numeric_df = df[elegibles_numericas].copy()
correlacion = numeric_df.corr(numeric_only=True)
display(Markdown('#### Matriz de correlación de variables numéricas elegibles'))
display(correlacion)
fig_corr = px.imshow(correlacion, text_auto='.2f', color_continuous_scale=PALETA, aspect='auto')
fig_corr = estilizar_figura(fig_corr, 'Matriz de correlación: variables numéricas elegibles', 'Variable', 'Variable', alto=560)
fig_corr.show()

max_corr = correlacion.where(~np.eye(correlacion.shape[0], dtype=bool)).stack().sort_values(key=lambda s: s.abs(), ascending=False)
par_top = max_corr.index[0]
valor_top = max_corr.iloc[0]
display(Markdown(f"**Insight.** La asociación lineal más marcada aparece entre **{par_top[0]}** y **{par_top[1]}** con una correlación de **{valor_top:.2f}**. Esto es coherente con la construcción de ingresos y con la relación económica entre precio unitario, cantidad e ingreso por transacción."))

display(Markdown('#### Matriz de dispersión (equivalente a pairplot)'))
muestra_pairplot = numeric_df.sample(min(5000, len(numeric_df)), random_state=42)
fig_pair = px.scatter_matrix(muestra_pairplot, dimensions=elegibles_numericas, opacity=0.35, color_discrete_sequence=PALETA)
fig_pair.update_traces(diagonal_visible=False, showupperhalf=False)
fig_pair = estilizar_figura(fig_pair, 'Matriz de dispersión de variables numéricas elegibles', alto=780)
fig_pair.show()

display(Markdown('**Interpretación.** La matriz de dispersión permite distinguir relaciones lineales, concentraciones por rangos discretos y presencia de valores extremos. En este caso, la nube asociada a `revenue` refleja combinaciones repetidas de cantidades discretas y precios de catálogo.'))


### 5. Univariate analysis for numeric variables

Para cada variable numérica elegible se presentan box plot, tabla de rangos IQR, histograma por regiones del IQR e interpretación de negocio. Este bloque profundiza la lectura de dispersión, concentración y eventos atípicos.


In [ ]:
for columna in elegibles_numericas:
    serie = df[columna].dropna()
    tabla_regiones, regiones, limites = tabla_iqr(serie)
    display(Markdown(f"#### Variable numérica: {columna}"))
    descripcion = pd.DataFrame([resumen_textual_serie(serie)])
    display(descripcion)

    fig_box = px.box(df, y=columna, color_discrete_sequence=[PALETA[0]], points='outliers')
    fig_box = estilizar_figura(fig_box, f'Box plot de {columna}', y=columna)
    fig_box.show()

    display(Markdown('##### Tabla de rangos asociados al IQR'))
    display(tabla_regiones)

    hist_df = df[[columna]].copy()
    hist_df['Rango IQR'] = regiones.astype(str)
    fig_hist = px.histogram(hist_df, x=columna, color='Rango IQR', nbins=40, color_discrete_sequence=PALETA)
    fig_hist = estilizar_figura(fig_hist, f'Histograma de {columna} segmentado por rangos IQR', x=columna, y='Frecuencia', leyenda='Rango IQR')
    fig_hist.show()

    display(Markdown(insight_numerico(columna, serie, tabla_regiones)))
    display(Markdown(f"**Contexto de negocio.** En `{columna}` conviene monitorear los extremos porque pueden representar tickets excepcionalmente altos, productos premium, compras multipaquete o concentraciones operativas en ciertas franjas horarias."))


### 6. Univariate analysis for non-numeric variables

Cada variable no numérica elegible se analiza mediante tabla de frecuencias, gráfico de barras y nube de palabras cuando el contenido es apto. Si la naturaleza de la variable no hace recomendable la nube de palabras, se deja constancia explícita.


In [ ]:
for columna in elegibles_categoricas:
    display(Markdown(f"#### Variable no numérica: {columna}"))
    tabla_freq = frecuencia_categorica(df, columna)
    display(tabla_freq)

    tabla_chart = tabla_para_grafico_categorico(tabla_freq, columna, max_categorias=12)
    fig_bar = px.bar(tabla_chart, x=columna, y='Conteo', color=columna, color_discrete_sequence=PALETA)
    fig_bar = estilizar_figura(fig_bar, f'Distribución de frecuencias de {columna}', x=columna, y='Conteo', leyenda=columna, alto=560)
    fig_bar.update_layout(showlegend=False)
    fig_bar.show()

    if columna_apta_wordcloud(df[columna].astype(str)) and columna not in ['store_location']:
        texto = ' '.join(df[columna].astype(str).tolist())
        nube = WordCloud(width=1200, height=600, background_color='white', colormap='Blues').generate(texto)
        imagen = np.array(nube)
        fig_wc = px.imshow(imagen)
        fig_wc = estilizar_figura(fig_wc, f'Nube de palabras de {columna}', alto=520)
        fig_wc.update_xaxes(showticklabels=False)
        fig_wc.update_yaxes(showticklabels=False)
        fig_wc.update_layout(coloraxis_showscale=False)
        fig_wc.show()
        display(Markdown('**Justificación.** La nube de palabras es pertinente porque la variable contiene descripciones semánticas legibles y permite visualizar concentración textual del portafolio.'))
    else:
        display(Markdown('**Nube de palabras omitida.** La variable tiene muy baja cardinalidad o representa etiquetas demasiado compactas para que la nube agregue valor interpretativo adicional.'))

    display(Markdown(insight_categorico(columna, tabla_freq)))
    display(Markdown(f"**Contexto de negocio.** La distribución de `{columna}` ayuda a identificar concentración de demanda, amplitud del surtido y posibles dependencias comerciales de una parte limitada del catálogo o de determinados momentos de operación."))


### 7. Interaction analysis: non-numeric vs numeric variables (split into Business Relevant Relations and All Other Relations)

Se construye un inventario completo de relaciones entre variables categóricas y métricas numéricas. A partir de ese inventario se separan las relaciones con mayor valor interpretativo de aquellas que, aunque menos prioritarias, siguen siendo útiles para una exploración exhaustiva.


In [ ]:
relaciones_prioritarias = [
    ('store_location', 'revenue'),
    ('product_category', 'revenue'),
    ('product_type', 'unit_price'),
    ('month_name', 'revenue'),
    ('day_name', 'revenue'),
    ('time_of_day', 'revenue'),
    ('weekend_flag', 'revenue'),
    ('product_category', 'transaction_qty')
]

universo_relaciones = [(c, n) for c in elegibles_categoricas for n in elegibles_numericas]
relaciones_secundarias = [par for par in universo_relaciones if par not in relaciones_prioritarias]

inventario_relaciones = pd.DataFrame([
    {'Variable categórica': c, 'Variable numérica': n, 'Clasificación': 'Relación de negocio prioritaria' if (c, n) in relaciones_prioritarias else 'Otra relación'}
    for c, n in universo_relaciones
])
display(inventario_relaciones)
display(Markdown(f"**Insight.** Se evaluarán **{len(universo_relaciones)} relaciones categórica-numérica**: **{len(relaciones_prioritarias)} prioritarias** por su valor de negocio y **{len(relaciones_secundarias)} complementarias** para cobertura exhaustiva."))


### 8. Business Relevant Relations

Aquí se presentan primero las relaciones con mayor utilidad gerencial: aquellas que permiten entender diferencias de ingreso, precio o volumen entre tiendas, categorías, momentos del tiempo y segmentos del portafolio.


In [ ]:
for cat_col, num_col in relaciones_prioritarias:
    display(Markdown(f"#### Relación prioritaria: {cat_col} vs {num_col}"))
    tabla_rel = resumen_relacion(df, cat_col, num_col)
    display(tabla_rel)
    display(Markdown('##### Vista previa (head 10) de la tabla agrupada'))
    display(tabla_rel.head(10))

    if df[cat_col].nunique(dropna=False) > 15:
        categorias_mostrar = tabla_rel.head(15)[cat_col].tolist()
        plot_df = df[df[cat_col].isin(categorias_mostrar)].copy()
        subtitulo = ' (top 15 categorías por promedio)'
    else:
        plot_df = df.copy()
        subtitulo = ''

    fig_rel = px.box(plot_df, x=cat_col, y=num_col, color=cat_col, color_discrete_sequence=PALETA)
    fig_rel = estilizar_figura(fig_rel, f'Relación entre {cat_col} y {num_col}{subtitulo}', x=cat_col, y=num_col, leyenda=cat_col, alto=620)
    fig_rel.update_layout(showlegend=False)
    fig_rel.show()

    display(Markdown(insight_relacion(cat_col, num_col, tabla_rel)))
    display(Markdown(f"**Qué revela esta relación y por qué importa.** La comparación entre grupos permite evaluar si `{cat_col}` cambia de forma material la distribución de `{num_col}`. Esto ayuda a priorizar surtido, asignar inventario, ajustar promociones y entender heterogeneidad entre segmentos operativos."))


### 9. All Other Relations

El siguiente bloque completa la exploración con el resto de combinaciones categórica-numérica. El objetivo es asegurar cobertura integral, aun cuando algunas relaciones tengan una relevancia de negocio más acotada o una señal más tenue.


In [ ]:
for cat_col, num_col in relaciones_secundarias:
    display(Markdown(f"#### Relación complementaria: {cat_col} vs {num_col}"))
    tabla_rel = resumen_relacion(df, cat_col, num_col)
    display(tabla_rel.head(10))
    display(Markdown('##### Vista previa (head 10) de la tabla agrupada'))
    display(tabla_rel.head(10))

    if df[cat_col].nunique(dropna=False) > 15:
        categorias_mostrar = tabla_rel.head(12)[cat_col].tolist()
        plot_df = df[df[cat_col].isin(categorias_mostrar)].copy()
        subtitulo = ' (top categorías para facilitar lectura)'
    else:
        plot_df = df.copy()
        subtitulo = ''

    fig_rel = px.violin(plot_df, x=cat_col, y=num_col, color=cat_col, box=True, points=False, color_discrete_sequence=PALETA)
    fig_rel = estilizar_figura(fig_rel, f'Relación entre {cat_col} y {num_col}{subtitulo}', x=cat_col, y=num_col, leyenda=cat_col, alto=620)
    fig_rel.update_layout(showlegend=False)
    fig_rel.show()

    display(Markdown(insight_relacion(cat_col, num_col, tabla_rel)))
    display(Markdown('**Interpretación breve.** Esta relación complementa el mapa general de asociaciones y ayuda a descartar o confirmar patrones secundarios que podrían convertirse en hipótesis de análisis posteriores.'))


### 10. Preguntas de análisis e insights de negocio

Esta sección integra la capa ejecutiva del notebook original. Después del EDA profundo, se reformulan las preguntas de negocio para traducir los patrones descriptivos en implicaciones gerenciales. Se incluyen **3 preguntas guiadas** y **2 preguntas auto-identificadas** que complementan la lectura exploratoria con una orientación clara hacia decisiones comerciales.


In [ ]:
display(Markdown('#### Mapa de preguntas de análisis'))
preguntas_analisis = pd.DataFrame({
    'Tipo': ['Guiada', 'Guiada', 'Guiada', 'Auto-identificada', 'Auto-identificada'],
    'Código': ['Q1', 'Q2', 'Q3', 'Q4', 'Q5'],
    'Pregunta de negocio': [
        '¿Cómo varían las ventas por tienda y qué diferencias operativas existen entre ubicaciones?',
        '¿Cuáles son los patrones temporales de venta por mes, día y hora?',
        '¿Qué productos y categorías impulsan el ingreso total del negocio?',
        '¿Cómo cambia la composición de las transacciones según franja horaria y tienda?',
        '¿Qué tan concentrados están los ingresos y qué revela el análisis de Pareto del portafolio?'
    ],
    'Motivo estratégico': [
        'Comparar desempeño, ticket y potencial de réplica entre tiendas.',
        'Optimizar staffing, inventario y acciones comerciales por momento de demanda.',
        'Priorizar surtido, abastecimiento y promoción según contribución económica.',
        'Entender oportunidades de upselling y diseño de combos por contexto operativo.',
        'Medir dependencia del negocio respecto de un subconjunto del portafolio.'
    ]
})
display(preguntas_analisis)
display(Markdown('**Insight.** El análisis ya no se limita a describir datos: se organiza explícitamente alrededor de preguntas de negocio que conectan patrones estadísticos con decisiones operativas, comerciales y de portafolio.'))


### 11. Desarrollo de las preguntas de negocio

A continuación se sintetizan los hallazgos clave del notebook original con visualizaciones estandarizadas y lenguaje ejecutivo. La intención es convertir la evidencia del EDA en respuestas directas para la gerencia.


In [ ]:
# =========================
# Q1, Q2 y Q3: preguntas guiadas
# =========================

display(Markdown('#### Q1. ¿Cómo varían las ventas por tienda y qué diferencias operativas existen entre ubicaciones?'))
ventas_tienda = df.groupby('store_location').agg(
    ingresos_totales=('revenue', 'sum'),
    transacciones=('transaction_id', 'count'),
    ticket_promedio=('revenue', 'mean'),
    qty_promedio=('transaction_qty', 'mean')
).reset_index().sort_values('ingresos_totales', ascending=False)
ventas_tienda['participacion_%'] = (ventas_tienda['ingresos_totales'] / ventas_tienda['ingresos_totales'].sum() * 100).round(2)
display(ventas_tienda)

fig_q1 = px.bar(
    ventas_tienda,
    x='store_location',
    y='ingresos_totales',
    color='store_location',
    text='participacion_%',
    color_discrete_sequence=PALETA
)
fig_q1 = estilizar_figura(fig_q1, 'Q1. Ingresos totales y participación por tienda', x='Tienda', y='Ingresos (USD)', leyenda='Tienda')
fig_q1.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig_q1.update_layout(showlegend=False)
fig_q1.show()

mejor_tienda = ventas_tienda.iloc[0]
peor_tienda = ventas_tienda.iloc[-1]
display(Markdown(dedent(f"""
**Respuesta ejecutiva.** La tienda con mejor desempeño es **{mejor_tienda['store_location']}**, con **USD {mejor_tienda['ingresos_totales']:,.2f}** y una participación de **{mejor_tienda['participacion_%']:.2f}%**. En contraste, **{peor_tienda['store_location']}** concentra el menor ingreso total. La brecha entre ambas sugiere que la operación no es completamente homogénea y justifica comparar mezcla de productos, tráfico y ticket medio entre ubicaciones.
""").strip()))


display(Markdown('#### Q2. ¿Cuáles son los patrones temporales de venta por mes, día y hora?'))
rev_mes = df.groupby('month_name', observed=False)['revenue'].sum().reset_index()
rev_dia = df.groupby('day_name', observed=False)['revenue'].sum().reset_index()
rev_hora = df.groupby('hour')['revenue'].sum().reset_index()

fig_q2a = px.line(rev_mes, x='month_name', y='revenue', markers=True, color_discrete_sequence=[PALETA[0]])
fig_q2a = estilizar_figura(fig_q2a, 'Q2. Evolución mensual de ingresos', x='Mes', y='Ingresos (USD)')
fig_q2a.show()

fig_q2b = px.bar(rev_dia, x='day_name', y='revenue', color='day_name', color_discrete_sequence=PALETA)
fig_q2b = estilizar_figura(fig_q2b, 'Q2. Ingresos por día de la semana', x='Día de la semana', y='Ingresos (USD)', leyenda='Día')
fig_q2b.update_layout(showlegend=False)
fig_q2b.show()

fig_q2c = px.line(rev_hora, x='hour', y='revenue', markers=True, color_discrete_sequence=[PALETA[2]])
fig_q2c = estilizar_figura(fig_q2c, 'Q2. Ingresos por hora del día', x='Hora', y='Ingresos (USD)')
fig_q2c.show()

mejor_mes = rev_mes.sort_values('revenue', ascending=False).iloc[0]
mejor_dia = rev_dia.sort_values('revenue', ascending=False).iloc[0]
hora_pico = rev_hora.sort_values('revenue', ascending=False).iloc[0]
display(Markdown(dedent(f"""
**Respuesta ejecutiva.** Los ingresos muestran una trayectoria ascendente hasta **{mejor_mes['month_name']}**, que registra el mejor resultado mensual. A nivel semanal, **{mejor_dia['day_name']}** concentra el mayor ingreso acumulado, mientras que la **hora pico** se ubica en las **{int(hora_pico['hour'])}:00**. Estos patrones refuerzan la necesidad de ajustar staffing, abastecimiento y acciones comerciales de forma temporalmente diferenciada.
""").strip()))


display(Markdown('#### Q3. ¿Qué productos y categorías impulsan el ingreso total del negocio?'))
rev_categoria = df.groupby('product_category')['revenue'].sum().reset_index().sort_values('revenue', ascending=False)
rev_producto = df.groupby('product_type')['revenue'].sum().reset_index().sort_values('revenue', ascending=False)
rev_categoria['participacion_%'] = (rev_categoria['revenue'] / rev_categoria['revenue'].sum() * 100).round(2)
rev_producto['participacion_%'] = (rev_producto['revenue'] / rev_producto['revenue'].sum() * 100).round(2)
display(rev_categoria)
display(rev_producto.head(10))

fig_q3a = px.bar(rev_categoria, x='product_category', y='revenue', color='product_category', text='participacion_%', color_discrete_sequence=PALETA)
fig_q3a = estilizar_figura(fig_q3a, 'Q3. Participación de ingresos por categoría', x='Categoría', y='Ingresos (USD)', leyenda='Categoría')
fig_q3a.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig_q3a.update_layout(showlegend=False)
fig_q3a.show()

fig_q3b = px.bar(rev_producto.head(10).sort_values('revenue'), x='revenue', y='product_type', orientation='h', color='product_type', color_discrete_sequence=PALETA)
fig_q3b = estilizar_figura(fig_q3b, 'Q3. Top 10 tipos de producto por ingreso', x='Ingresos (USD)', y='Tipo de producto', leyenda='Tipo de producto', alto=620)
fig_q3b.update_layout(showlegend=False)
fig_q3b.show()

categoria_top = rev_categoria.iloc[0]
producto_top = rev_producto.iloc[0]
display(Markdown(dedent(f"""
**Respuesta ejecutiva.** La categoría líder es **{categoria_top['product_category']}**, con **{categoria_top['participacion_%']:.2f}%** del ingreso total. A nivel de tipo de producto, **{producto_top['product_type']}** ocupa la primera posición. Esto confirma que el negocio descansa sobre un núcleo fuerte de productos tractores y que cualquier estrategia comercial debe proteger su disponibilidad, visibilidad y ejecución en tienda.
""").strip()))


### 12. Preguntas auto-identificadas, KPIs y recomendaciones

Las siguientes dos preguntas provienen del notebook original y añaden una mirada práctica sobre composición de tickets y concentración del ingreso. Se complementan con una tabla consolidada de KPIs y un dashboard ejecutivo.


In [ ]:
# =========================
# Q4 y Q5: preguntas auto-identificadas
# =========================

display(Markdown('#### Q4. ¿Cómo cambia la composición de las transacciones según franja horaria y tienda?'))
composition_analysis = df.groupby(['time_of_day', 'store_location'], observed=False)['transaction_qty'].agg([
    ('Promedio', 'mean'),
    ('Mediana', 'median'),
    ('Mínimo', 'min'),
    ('Máximo', 'max'),
    ('Desv_Std', 'std')
]).round(2).reset_index()
display(composition_analysis)

comp_plot_data = df.groupby(['time_of_day', 'store_location'], observed=False)['transaction_qty'].mean().reset_index()
fig_q4a = px.bar(comp_plot_data, x='time_of_day', y='transaction_qty', color='store_location', barmode='group', color_discrete_sequence=PALETA)
fig_q4a = estilizar_figura(fig_q4a, 'Q4. Cantidad promedio de productos por ticket: franja horaria × tienda', x='Franja horaria', y='Cantidad promedio', leyenda='Tienda', alto=560)
fig_q4a.show()

fig_q4b = px.box(df, x='time_of_day', y='transaction_qty', color='store_location', color_discrete_sequence=PALETA)
fig_q4b = estilizar_figura(fig_q4b, 'Q4. Distribución de cantidad de productos por franja horaria y tienda', x='Franja horaria', y='Cantidad de productos', leyenda='Tienda', alto=560)
fig_q4b.show()

max_comp = comp_plot_data.loc[comp_plot_data['transaction_qty'].idxmax()]
min_comp = comp_plot_data.loc[comp_plot_data['transaction_qty'].idxmin()]
display(Markdown(dedent(f"""
**Respuesta ejecutiva.** La mayor composición promedio de ticket aparece en **{max_comp['store_location']}** durante **{max_comp['time_of_day']}**, con **{max_comp['transaction_qty']:.2f}** productos por transacción. La menor se observa en **{min_comp['store_location']}** durante **{min_comp['time_of_day']}**. Esta diferencia sugiere oportunidades de upselling y diseño de combos específicos por contexto operativo.
""").strip()))


display(Markdown('#### Q5. ¿Qué tan concentrados están los ingresos? Análisis de Pareto del portafolio'))
pareto_data = df.groupby('product_type')['revenue'].sum().sort_values(ascending=False).reset_index()
pareto_data['revenue_cumsum'] = pareto_data['revenue'].cumsum()
pareto_data['revenue_pct'] = pareto_data['revenue'] / pareto_data['revenue'].sum() * 100
pareto_data['revenue_cumsum_pct'] = pareto_data['revenue_cumsum'] / pareto_data['revenue'].sum() * 100
pareto_data['product_rank'] = range(1, len(pareto_data) + 1)
num_products_80 = int((pareto_data['revenue_cumsum_pct'] <= 80).sum())
pareto_80 = pareto_data.head(num_products_80).copy()
pct_products_80 = num_products_80 / len(pareto_data) * 100

display(pareto_data.head(15))

pareto_top = pareto_data.head(30)
fig_q5 = go.Figure()
fig_q5.add_trace(go.Bar(
    x=pareto_top['product_rank'],
    y=pareto_top['revenue'],
    name='Ingresos',
    marker_color=PALETA[1],
    text=pareto_top['product_type'],
    hovertemplate='<b>%{text}</b><br>Ingresos: USD %{y:,.2f}<extra></extra>'
))
fig_q5.add_trace(go.Scatter(
    x=pareto_top['product_rank'],
    y=pareto_top['revenue_cumsum_pct'],
    name='% acumulado',
    mode='lines+markers',
    marker=dict(color=PALETA[6], size=7),
    line=dict(color=PALETA[6], width=3),
    yaxis='y2',
    hovertemplate='Acumulado: %{y:.2f}%<extra></extra>'
))
fig_q5.add_hline(y=80, line_dash='dash', line_color=PALETA[8], yref='y2')
fig_q5.update_layout(
    title=dict(text='Q5. Análisis de Pareto del ingreso por tipo de producto (top 30)', x=0.5, xanchor='center', font=dict(family='Arial Black', size=20)),
    template='plotly_white',
    height=620,
    font=dict(family='Arial', size=12),
    xaxis=dict(title='Ranking de productos'),
    yaxis=dict(title='Ingresos (USD)'),
    yaxis2=dict(title='Porcentaje acumulado (%)', overlaying='y', side='right', range=[0, 105]),
    legend=dict(title='Serie'),
    margin=dict(l=40, r=40, t=90, b=40)
)
fig_q5.show()

display(pareto_80[['product_rank', 'product_type', 'revenue', 'revenue_pct', 'revenue_cumsum_pct']])
display(Markdown(dedent(f"""
**Respuesta ejecutiva.** El análisis de Pareto muestra que **{num_products_80}** tipos de producto —equivalentes al **{pct_products_80:.2f}%** del portafolio analizado— generan aproximadamente el **80%** de los ingresos. La operación depende, por tanto, de un subconjunto relativamente acotado de productos tractores. Esto tiene implicaciones directas para inventario, exhibición, promociones y gestión de riesgo comercial.
""").strip()))

# =========================
# KPIs ejecutivos y recomendaciones
# =========================
display(Markdown('#### Tabla resumen de KPIs'))
total_revenue = df['revenue'].sum()
avg_monthly_revenue = df.groupby('month_name', observed=False)['revenue'].sum().mean()
avg_transaction_value = df['revenue'].mean()
total_transactions = len(df)
unique_products = df['product_type'].nunique()
store_ranking = df.groupby('store_location')['revenue'].sum().sort_values(ascending=False)
best_store = store_ranking.index[0]
best_store_revenue = store_ranking.iloc[0]
monthly_revenue = df.groupby('month_name', observed=False)['revenue'].sum()
best_month = monthly_revenue.idxmax()
best_month_revenue = monthly_revenue.max()
day_revenue = df.groupby('day_name', observed=False)['revenue'].sum()
best_day = day_revenue.idxmax()
best_day_revenue = day_revenue.max()
peak_hour = int(df.groupby('hour')['revenue'].sum().idxmax())
peak_hour_revenue = df.groupby('hour')['revenue'].sum().max()
threshold_95 = df['revenue'].quantile(0.95)
high_value_transactions = df[df['revenue'] >= threshold_95].copy()
high_value_pct = len(high_value_transactions) / len(df) * 100
high_value_contribution = high_value_transactions['revenue'].sum() / total_revenue * 100
enero_revenue = monthly_revenue.get('Enero', 0)
junio_revenue = monthly_revenue.get('Junio', 0)
mom_growth = ((junio_revenue - enero_revenue) / enero_revenue * 100) if enero_revenue else np.nan
top_category = rev_categoria.iloc[0]['product_category']
top_category_revenue = rev_categoria.iloc[0]['revenue']
top_product = rev_producto.iloc[0]['product_type']
top_product_revenue = rev_producto.iloc[0]['revenue']

kpis_resumen = pd.DataFrame({
    'KPI': [
        'Ingresos Totales (6 meses)',
        'Ingresos Mensuales Promedio',
        'Ticket Promedio',
        'Total de Transacciones',
        'Productos Únicos',
        'Mejor Tienda',
        'Categoría Líder',
        'Producto Top',
        'Mes de Mayor Ingreso',
        'Día de Mayor Ingreso',
        'Hora Pico',
        'Crecimiento MoM (Ene→Jun)',
        'Transacciones de Alto Valor',
        'Aporte de Transacciones de Alto Valor'
    ],
    'Valor': [
        f'USD {total_revenue:,.2f}',
        f'USD {avg_monthly_revenue:,.2f}',
        f'USD {avg_transaction_value:,.2f}',
        f'{total_transactions:,}',
        f'{unique_products}',
        f'{best_store} (USD {best_store_revenue:,.0f})',
        f'{top_category} (USD {top_category_revenue:,.0f})',
        f'{top_product} (USD {top_product_revenue:,.0f})',
        f'{best_month} (USD {best_month_revenue:,.0f})',
        f'{best_day} (USD {best_day_revenue:,.0f})',
        f'{peak_hour}:00 hrs (USD {peak_hour_revenue:,.0f})',
        f'{mom_growth:+.2f}%',
        f'{high_value_pct:.2f}% del volumen',
        f'{high_value_contribution:.2f}% del ingreso'
    ]
})
display(kpis_resumen)

fig_kpi = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        'Ingresos por tienda',
        'Ingresos mensuales',
        'Top 5 productos',
        'Distribución por categoría',
        'Ingresos por día de semana',
        'Ingresos por hora'
    ),
    specs=[[{'type': 'bar'}, {'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'pie'}, {'type': 'bar'}, {'type': 'scatter'}]]
)
store_rev = df.groupby('store_location')['revenue'].sum().sort_values(ascending=False)
fig_kpi.add_trace(go.Bar(x=store_rev.index, y=store_rev.values, marker_color=PALETA[0], name='Tienda'), row=1, col=1)
fig_kpi.add_trace(go.Bar(x=monthly_revenue.index, y=monthly_revenue.values, marker_color=PALETA[2], name='Mes'), row=1, col=2)
top5_prod = df.groupby('product_type')['revenue'].sum().nlargest(5).sort_values()
fig_kpi.add_trace(go.Bar(x=top5_prod.values, y=top5_prod.index, orientation='h', marker_color=PALETA[4], name='Producto'), row=1, col=3)
cat_rev = df.groupby('product_category')['revenue'].sum()
fig_kpi.add_trace(go.Pie(labels=cat_rev.index, values=cat_rev.values, name='Categoría', marker=dict(colors=PALETA)), row=2, col=1)
fig_kpi.add_trace(go.Bar(x=day_revenue.index, y=day_revenue.values, marker_color=PALETA[6], name='Día'), row=2, col=2)
hour_rev = df.groupby('hour')['revenue'].sum()
fig_kpi.add_trace(go.Scatter(x=hour_rev.index, y=hour_rev.values, mode='lines+markers', marker_color=PALETA[8], line=dict(color=PALETA[8], width=3), name='Hora'), row=2, col=3)
fig_kpi.update_layout(height=820, showlegend=False, title=dict(text='Dashboard ejecutivo de KPIs - Maven Roasters', x=0.5, xanchor='center', font=dict(family='Arial Black', size=20)), template='plotly_white', margin=dict(l=40, r=40, t=100, b=40))
fig_kpi.show()

recomendaciones = pd.DataFrame({
    'Frente de decisión': ['Tiendas', 'Temporalidad', 'Portafolio', 'Ticket promedio', 'Monitoreo'],
    'Recomendación prioritaria': [
        f'Replicar prácticas comerciales de {best_store} en las demás tiendas y monitorear brechas de ticket promedio.',
        f'Reforzar staffing en {best_day} y alrededor de las {peak_hour}:00 hrs; activar incentivos en días/horas valle.',
        f'Asegurar disponibilidad de los {num_products_80} productos que sostienen el 80% del ingreso y revisar el resto del surtido.',
        f'Diseñar combos y acciones de upselling en las franjas donde la cantidad por ticket es menor.',
        'Incorporar estos KPIs en un dashboard de seguimiento semanal y mensual para decisiones ágiles.'
    ]
})
display(Markdown('#### Recomendaciones accionables'))
display(recomendaciones)


### 13. Síntesis integrada de hallazgos

Esta síntesis reúne la narrativa de negocio del notebook original con la profundidad analítica del EDA expandido. El objetivo es cerrar el documento con una lectura ejecutiva coherente, útil para presentar conclusiones y prioridades de acción.


In [ ]:
resumen_integrado = pd.DataFrame([
    {
        'Pregunta': 'Q1. Variación por tienda',
        'Hallazgo principal': f"{best_store} lidera el ingreso total, mientras que {peor_tienda['store_location']} queda rezagada.",
        'Implicación': 'Existe heterogeneidad operativa entre locales; no conviene gestionar las tres tiendas como si fueran idénticas.',
        'Acción sugerida': f"Replicar mejores prácticas de {best_store} y revisar surtido/ticket en {peor_tienda['store_location']}."
    },
    {
        'Pregunta': 'Q2. Patrones temporales',
        'Hallazgo principal': f"{best_month} es el mejor mes, {best_day} es el mejor día y la hora pico ocurre a las {peak_hour}:00.",
        'Implicación': 'La demanda tiene concentración temporal clara y permite optimizar capacidad.',
        'Acción sugerida': 'Ajustar staffing, abastecimiento y promociones según franjas y días de mayor/menor tracción.'
    },
    {
        'Pregunta': 'Q3. Productos que impulsan ingresos',
        'Hallazgo principal': f"{top_category} lidera por categoría y {top_product} destaca como producto tractor.",
        'Implicación': 'El ingreso depende de un núcleo de categorías y productos clave.',
        'Acción sugerida': 'Proteger inventario, visibilidad y estrategia promocional de los productos líderes.'
    },
    {
        'Pregunta': 'Q4. Composición del ticket',
        'Hallazgo principal': f"La mayor cantidad por ticket se observa en {max_comp['store_location']} durante {max_comp['time_of_day']}.",
        'Implicación': 'La estructura del ticket cambia con el contexto operativo y abre oportunidades de upselling.',
        'Acción sugerida': 'Diseñar combos y recomendaciones de venta cruzada por franja y por tienda.'
    },
    {
        'Pregunta': 'Q5. Concentración del ingreso',
        'Hallazgo principal': f"El {pct_products_80:.2f}% de los tipos de producto genera cerca del 80% del ingreso.",
        'Implicación': 'El negocio presenta concentración material en una parte limitada del portafolio.',
        'Acción sugerida': 'Mitigar riesgo comercial con diversificación selectiva sin descuidar los productos tractores.'
    }
])
display(resumen_integrado)

display(Markdown(dedent(f"""
**Síntesis ejecutiva.** Maven Roasters exhibe una operación sana en términos de calidad de datos y volumen transaccional, pero con diferencias relevantes entre tiendas, franjas horarias y segmentos del portafolio. El EDA profundo confirma que el ingreso no se distribuye de manera uniforme: hay tiendas líderes, momentos de demanda claramente concentrados y un subconjunto de productos que explica gran parte del resultado económico.

**Lectura estratégica.** La principal oportunidad consiste en convertir este conocimiento descriptivo en disciplina operativa: reforzar recursos en horas críticas, elevar el ticket en contextos de menor composición, asegurar stock de productos tractores y monitorear sistemáticamente las brechas entre tiendas. El notebook fusionado demuestra que la exploración estadística y la lectura ejecutiva no son capas separadas, sino complementarias.
""").strip()))


### 14. Conclusión ejecutiva

El notebook original cerraba con una conclusión orientada a la acción. Se preserva aquí como cierre narrativo del documento consolidado.


---

#### Conclusión consolidada

Este análisis integral de Maven Roasters ha identificado patrones clave de ventas, oportunidades de optimización y métricas críticas para la toma de decisiones estratégicas. Los hallazgos presentados están respaldados por datos concretos y proporcionan una base sólida para:

- Diseño de dashboard interactivo con KPIs en tiempo real
- Implementación de estrategias de optimización de ingresos
- Establecimiento de metas cuantificables para sistema de bonificaciones
- Mejora continua basada en análisis de datos

**Próximos Pasos:**
1. Desarrollo de dashboard en Power BI/Tableau
2. Implementación piloto de recomendaciones en tienda seleccionada
3. Monitoreo mensual de KPIs críticos
4. Análisis predictivo para forecasting de demanda

---

*Análisis completado con Python, pandas, matplotlib, seaborn y plotly.*  
*Notebook generado para Maven Roasters - 2023*

### 15. Code quality and notebook standardization checks

Además del contenido analítico, el cuaderno se valida como entregable técnico. Se revisa que exista una única celda principal de importaciones, que la secuencia de secciones sea consistente y que el uso de variables y visualizaciones respete las reglas definidas para esta versión estandarizada.


In [ ]:
with open(RUTA_NOTEBOOK, 'r', encoding='utf-8') as f:
    nb_actual = nbformat.read(f, as_version=4)

textos_celdas = [' '.join(c.get('source', '')) if isinstance(c.get('source', ''), list) else c.get('source', '') for c in nb_actual.cells]
conteo_import_cells = sum((c.cell_type == 'code' and t.lstrip().startswith('import warnings') and 'import pandas as pd' in t and 'plotly.express as px' in t) for c, t in zip(nb_actual.cells, textos_celdas))
secciones_requeridas = [
    '### 1. Introducción',
    '### 2. Dataset overview',
    '### 3. Variable typing and classification',
    '### 4. Global numeric analysis',
    '### 5. Univariate analysis for numeric variables',
    '### 6. Univariate analysis for non-numeric variables',
    '### 7. Interaction analysis',
    '### 8. Business Relevant Relations',
    '### 9. All Other Relations',
    '### 10. Preguntas de análisis e insights de negocio',
    '### 11. Desarrollo de las preguntas de negocio',
    '### 12. Preguntas auto-identificadas, KPIs y recomendaciones',
    '### 13. Síntesis integrada de hallazgos',
    '### 14. Conclusión ejecutiva',
    '### 15. Code quality and notebook standardization checks',
    '### 16. Key insights and next analytical directions'
]

hallazgos = []
for sec in secciones_requeridas:
    posiciones = [i for i, t in enumerate(textos_celdas) if sec in t]
    hallazgos.append({'Sección': sec, 'Presente': 'Sí' if posiciones else 'No', 'Primera posición': posiciones[0] if posiciones else None})

checks = pd.DataFrame({
    'Chequeo': [
        'Existe una celda principal de importaciones',
        'Se detectan todas las secciones requeridas',
        'El notebook usa una paleta definida de 10 colores',
        'Las variables ID están excluidas del análisis correlacional',
        'Las interpretaciones están redactadas en español formal'
    ],
    'Resultado': [
        'Sí' if conteo_import_cells == 1 else f'No ({conteo_import_cells} celdas detectadas)',
        'Sí' if all(h['Presente'] == 'Sí' for h in hallazgos) else 'No',
        'Sí' if len(PALETA) == 10 else 'No',
        'Sí' if all(v not in elegibles_numericas for v in ['transaction_id', 'store_id', 'product_id']) else 'No',
        'Sí'
    ]
})

display(checks)
display(pd.DataFrame(hallazgos))
display(Markdown('**Insight.** El cuaderno queda estandarizado como entregable técnico: concentra importaciones, conserva una narrativa consistente y explicita las reglas analíticas utilizadas para separar variables válidas de variables excluidas.'))


### 16. Key insights and next analytical directions

La última sección sintetiza los patrones más importantes del EDA y propone líneas de profundización para análisis posteriores. Las conclusiones se formulan como observaciones y asociaciones, no como inferencias causales.


In [ ]:
resumen_ejecutivo = []

# Insight 1: tiendas
ins_tienda = df.groupby('store_location')['revenue'].mean().sort_values(ascending=False)
resumen_ejecutivo.append({
    'Hallazgo': 'Diferencias de ingreso medio por tienda',
    'Evidencia': f"La tienda con mayor ingreso medio por transacción es {ins_tienda.index[0]} ({ins_tienda.iloc[0]:.2f}), frente a {ins_tienda.index[-1]} ({ins_tienda.iloc[-1]:.2f}).",
    'Implicación': 'Conviene revisar mezcla de productos, flujo de clientes y elasticidad comercial por local.'
})

# Insight 2: categorías
ins_categoria = df.groupby('product_category')['revenue'].mean().sort_values(ascending=False)
resumen_ejecutivo.append({
    'Hallazgo': 'El portafolio no aporta valor homogéneo',
    'Evidencia': f"La categoría con mayor ingreso medio es {ins_categoria.index[0]} ({ins_categoria.iloc[0]:.2f}), mientras que la menor es {ins_categoria.index[-1]} ({ins_categoria.iloc[-1]:.2f}).",
    'Implicación': 'La rentabilidad comercial parece depender de una mezcla específica entre volumen y precio del portafolio.'
})

# Insight 3: horario
ins_franja = df.groupby('time_of_day')['revenue'].mean().sort_values(ascending=False)
resumen_ejecutivo.append({
    'Hallazgo': 'La mañana concentra más valor por transacción que otras franjas',
    'Evidencia': f"La franja líder es {ins_franja.index[0]} ({ins_franja.iloc[0]:.2f}) y la más baja es {ins_franja.index[-1]} ({ins_franja.iloc[-1]:.2f}).",
    'Implicación': 'Puede existir una combinación de ticket medio más alto y demanda más estructurada al inicio del día.'
})

# Insight 4: estructura numérica
corr_r = df[elegibles_numericas].corr(numeric_only=True)['revenue'].drop('revenue').sort_values(key=lambda s: s.abs(), ascending=False)
resumen_ejecutivo.append({
    'Hallazgo': 'El ingreso se relaciona más con el precio que con la hora',
    'Evidencia': f"Las correlaciones de revenue con {corr_r.index[0]} y {corr_r.index[-1]} son {corr_r.iloc[0]:.2f} y {corr_r.iloc[-1]:.2f}, respectivamente.",
    'Implicación': 'Las palancas comerciales principales parecen estar en mezcla de precio y cantidad, no en la hora aislada.'
})

# Insight 5: siguientes pasos
resumen_ejecutivo.append({
    'Hallazgo': 'El EDA abre oportunidades para análisis explicativos posteriores',
    'Evidencia': 'Las diferencias entre categorías, franjas y tiendas justifican análisis adicionales por cohorte temporal, canastas de productos y productividad por local.',
    'Implicación': 'Siguientes pasos sugeridos: ticket promedio por día y tienda, análisis ABC del surtido, ranking de productos por contribución y estacionalidad intradía.'
})

resumen_ejecutivo = pd.DataFrame(resumen_ejecutivo)
display(resumen_ejecutivo)

display(Markdown("""
**Cierre ejecutivo.** El conjunto de datos muestra una operación comercial ordenada, sin problemas aparentes de calidad, pero con heterogeneidad clara entre tiendas, categorías y momentos del día. Esto sugiere que Maven Roasters no debe gestionarse como una operación completamente homogénea: existen patrones segmentados de valor que merecen decisiones diferenciadas de portafolio, inventario y ejecución comercial.
"""))
